In [ ]:
import pyttsx3
from time import sleep

In [ ]:
# engine = pyttsx3.init() # object creation

In [ ]:
# engine = pyttsx3.init()
# voices = engine.getProperty('voices')

# for ivoice, voice in enumerate(voices):
#     print(f"# {ivoice}")
#     print(f"ID: {voice.id}")
#     print(f"Name: {voice.name}")
#     print(f"Languages: {voice.languages}")
#     print(f"Gender: {voice.gender}")
#     print(f"Age: {voice.age}")
#     print("----------")

In [ ]:
# """ RATE"""
# rate = engine.getProperty('rate')   # getting details of current speaking rate
# print(rate)                        #printing current voice rate
# engine.setProperty('rate', 125)     # setting up new voice rate

In [ ]:
# """VOLUME"""
# volume = engine.getProperty('volume')   #getting to know current volume level (min=0 and max=1)
# print(volume)                          #printing current volume level
# engine.setProperty('volume',1.0)    # setting up volume level  between 0 and 1

In [ ]:
# """VOICE"""
# voices = engine.getProperty('voices')       #getting details of current voice
# #engine.setProperty('voice', voices[0].id)  #changing index, changes voices. 0 for male
# engine.setProperty('voice', voices[1].id)   #changing index, changes voices. 1 for female

In [ ]:
# engine.say("I will speak this text")

In [ ]:
# engine.say("Hello World!")
# engine.say('My current speaking rate is ' + str(rate))
# engine.runAndWait()
# engine.stop()

In [ ]:
# """Saving Voice to a file"""
# # On linux make sure that 'espeak' and 'ffmpeg' are installed
# engine.save_to_file('Hello World', 'test.mp3')
# engine.runAndWait()

In [ ]:
# class approach

In [ ]:

class TextToSpeech:
    def __init__(self, rate=125, volume=1.0, voice_index=0):
        self.engine = pyttsx3.init()
        self.rate = rate
        self.volume = volume
        self.voice_index = voice_index
        self.setup()

    def setup(self):
        self.engine.setProperty('rate', self.rate)
        self.engine.setProperty('volume', self.volume)
        voices = self.engine.getProperty('voices')
        self.engine.setProperty('voice', voices[self.voice_index].id)

    def speak(self, text):
        print(text)  # Echo the text to the screen
        self.engine.say(text)
        self.engine.runAndWait()

    def speak_trial_info(self, day_number, trial_number, reward_given=False, reward_amount=None, trainer=1):
        reward_text = "no reward"
        if reward_given and reward_amount in [70, 30]:
            reward_text = f"reward of {reward_amount}"
        
        text = f"Day {day_number}, trial {trial_number}, {reward_text}, trainer {trainer}"
        self.speak(text)

In [ ]:
# Example usage
tts = TextToSpeech(rate=20, volume=3.0)

In [ ]:
# RaspberryPi

# sudo apt-get update
# sudo apt-get install espeak ffmpeg

In [ ]:
for day in range(1, 7):  # Loop over days 1 to 6
    for trial in range(1, 11):  # Loop over trials 1 to 10
        # Example conditionals for demonstration
        reward_given = trial % 2 == 0  # Reward given on even trials
        reward_amount = 70 if day % 2 == 0 else 30  # 70 on even days, 30 on odd
        trainer = 1 if day < 4 else 2  # Trainer 1 for days 1-3, Trainer 2 for days 4-6
        tts.speak_trial_info(day, trial, reward_given, reward_amount, trainer)
        sleep(1.0)
        

In [ ]:
from gtts import gTTS
import os
#import tempfile
import socket
import hashlib

In [ ]:
class TextToSpeechGoogle:
    def __init__(self, lang='en', cache_dir='gtts_cache'):
        self.lang = lang
        self.cache_dir = cache_dir
        if not os.path.exists(self.cache_dir):
            os.makedirs(self.cache_dir)

    def is_connected(self):
        try:
            socket.create_connection(("www.google.com", 80), timeout=5)
            return True
        except OSError:
            return False

    def get_cache_filename(self, text):
        # Create a hash of the text to generate a unique filename
        text_hash = hashlib.md5(text.encode('utf-8')).hexdigest()
        return os.path.join(self.cache_dir, f"{text_hash}.mp3")

    def speak(self, text):
        cache_filename = self.get_cache_filename(text)
        # Check if the audio file already exists in the cache
        if not os.path.exists(cache_filename):
            if not self.is_connected():
                print("No internet connection available and no cached file.")
                return
            print("Generating new audio file...")
            tts = gTTS(text=text, lang=self.lang, slow=False)
            tts.save(cache_filename)
        else:
            print("Using cached audio file.")
        
        print(text)  # Echo the text to the screen
        os.system(f"mpg321 {cache_filename} || afplay {cache_filename} || play {cache_filename}")

    def speak_trial_info(self, day_number, trial_number, reward_given=False, reward_amount=None, trainer=1):
        reward_text = "no reward"
        if reward_given and reward_amount in [70, 30]:
            reward_text = f"reward of {reward_amount}"
        
        text = f"Day {day_number}, trial {trial_number}, {reward_text}, trainer {trainer}"
        self.speak(text)

In [ ]:
tts_google = TextToSpeechGoogle()
tts_google.speak("Hello, this is a test.")

In [ ]:
for day in range(1, 7):  # Loop over days 1 to 6
    for trial in range(1, 11):  # Loop over trials 1 to 10
        # Example conditionals for demonstration
        reward_given = trial % 2 == 0  # Reward given on even trials
        reward_amount = 70 if day % 2 == 0 else 30  # 70 on even days, 30 on odd
        trainer = 1 if day < 4 else 2  # Trainer 1 for days 1-3, Trainer 2 for days 4-6
        tts_google.speak_trial_info(day, trial, reward_given, reward_amount, trainer)
        sleep(1.0)